[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herrrickshaw/global-stock-screener/blob/main/notebooks/05_names_and_manual.ipynb)


# 🏷️ Names → Tickers  &  ✍️ Manual Buy/Hold/Sell
Two ways to get a **BUY / HOLD / SELL** call (with an **IMPAIRED** flag for names
unlikely to ever recover):
1. **By company name** — paste names from a tax/holdings statement; they're matched to tickers.
2. **Manual entry** — type a ticker + price/P/E inputs and decide, no dataset needed.

*Research only — not investment advice.*


In [ ]:
# Setup — clone the repo + pull the central LFS data, then establish + verify
# the initial conditions (STM seeds derived from LTM, serving views built).
!pip install -q lmdb pyarrow scikit-learn nsepython yfinance vaderSentiment openpyxl
!git lfs install
%cd /content
![ -d global-stock-screener ] || git clone --depth 1 https://github.com/herrrickshaw/global-stock-screener.git
%cd /content/global-stock-screener
!git lfs pull
import os; os.environ['BHAV_CACHE'] = '/content/cache'
import preflight
preflight.ensure(['IN', 'US'])   # make initial conditions true
assert preflight.check(['IN', 'US'])[0], 'initial conditions not met — see report above'


## 1️⃣ By company NAME (tax statement shows names, not tickers)


In [ ]:
import portfolio_evaluator as pe
names = ['Reliance Industries Ltd', 'Infosys', 'HDFC Bank', 'Wheels India']  # ← edit
tickers = pe.resolve_names(names, market='IN')
pe.evaluate(tickers, market='IN')


### …or upload a file whose only identifier is the company name


In [ ]:
from google.colab import files            # Colab only
up = files.upload()                        # a CSV/XLSX with a Name column
pe.evaluate_from_file(next(iter(up)), market='IN')


## 2️⃣ MANUAL entry — type ticker + price inputs
Fill one dict per stock. Only `price` + `pe` are required; adding `high_52w`,
`low_52w`, `sma200`, `ret_1y`, `roe` sharpens the call (esp. the IMPAIRED check).


In [ ]:
import portfolio_evaluator as pe
my_stocks = [
  {'ticker':'STOCK_A', 'price':120, 'pe':12, 'high_52w':125, 'low_52w':70,
   'sma200':100, 'ret_1y':45, 'roe':22},
  {'ticker':'STOCK_B', 'price':8,  'pe':-3, 'high_52w':60,  'low_52w':7,
   'sma200':30,  'ret_1y':-75, 'roe':-12, 'liquidity':'Low'},
]
pe.evaluate_manual(my_stocks, med_pe=22)   # med_pe = your market's typical P/E


**Columns:** `Signal` (BUY/HOLD/SELL), `Flags` (IMPAIRED = unlikely to recover,
WEAK-RECOVERY = at risk), `Score` (0–100 composite), `Recovery` (0–100), `Rationale`.
